# BanglaBERT Transformer Baseline - Binary Datasets (Single Run, Loop Version)

This notebook trains and evaluates **BanglaBERT** on all three binary datasets in **one run**:

- `ben_sarc_binary`
- `banglasarc_binary`
- `banglasarc3_binary`

## What this notebook does
- loops through all binary datasets automatically
- trains a separate BanglaBERT model for each dataset
- evaluates on validation and test splits
- saves:
  - summary metrics
  - per-dataset metrics
  - confusion matrices
  - classification reports
  - per-example predictions with probabilities
  - training metadata

## What this notebook does **not** do
- multi-seed experiments
- FGM / adversarial training
- cross-dataset transfer
- calibration analysis

Those will be handled in later notebooks.


In [1]:

from pathlib import Path
import json
import math
import random
import shutil
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

print("Torch version:", torch.__version__)
try:
    import transformers
    print("Transformers version:", transformers.__version__)
except Exception as e:
    print("Could not read transformers version:", e)

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("Detected device:", DEVICE)


/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch version: 2.11.0
Transformers version: 5.3.0
Detected device: mps


In [2]:

# =========================
# CONFIG
# =========================

SEED = 42
MODEL_NAME = "csebuetnlp/banglabert"

MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

USE_FP16 = torch.cuda.is_available()   # only works on CUDA
USE_BF16 = False
USE_NORMALIZATION = False              # keep False for baseline notebook
OVERWRITE_EXISTING_CHECKPOINTS = True
SAVE_TEST_PREDICTIONS = True
SAVE_VAL_PREDICTIONS = True

DATASET_NAMES = [
    "ben_sarc_binary",
    "banglasarc_binary",
    "banglasarc3_binary",
]

LABEL_COL = "label_binary"
TEXT_COL = "text"
POSITIVE_LABEL = 1

print("Model:", MODEL_NAME)
print("Datasets:", DATASET_NAMES)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Max length:", MAX_LENGTH)


Model: csebuetnlp/banglabert
Datasets: ['ben_sarc_binary', 'banglasarc_binary', 'banglasarc3_binary']
Epochs: 3
Batch size: 8
Max length: 128


In [3]:

# =========================
# REPRODUCIBILITY
# =========================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    torch.use_deterministic_algorithms(False)
except Exception:
    pass

print("Global seed set to:", SEED)


Global seed set to: 42


In [4]:

# =========================
# PATHS
# =========================

ROOT = Path("..")
SPLITS = ROOT / "01_data" / "interim" / "splits"
CHECKPOINTS = ROOT / "03_models" / "checkpoints"
TABLES = ROOT / "04_outputs" / "tables"
PREDICTIONS = ROOT / "04_outputs" / "predictions"
REPORTS = ROOT / "04_outputs" / "reports"
RUN_LOGS = ROOT / "04_outputs" / "run_logs"

for p in [CHECKPOINTS, TABLES, PREDICTIONS, REPORTS, RUN_LOGS]:
    p.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT.resolve())
print("SPLITS:", SPLITS.resolve())
print("CHECKPOINTS:", CHECKPOINTS.resolve())
print("TABLES:", TABLES.resolve())
print("PREDICTIONS:", PREDICTIONS.resolve())
print("REPORTS:", REPORTS.resolve())
print("RUN_LOGS:", RUN_LOGS.resolve())


ROOT: /Users/sefayet/Desktop/Github/Machine_Learning-Deep_Learning-Courses-and-Paper-Publish/Thesis_Papers/Sarcasm_detection
SPLITS: /Users/sefayet/Desktop/Github/Machine_Learning-Deep_Learning-Courses-and-Paper-Publish/Thesis_Papers/Sarcasm_detection/01_data/interim/splits
CHECKPOINTS: /Users/sefayet/Desktop/Github/Machine_Learning-Deep_Learning-Courses-and-Paper-Publish/Thesis_Papers/Sarcasm_detection/03_models/checkpoints
TABLES: /Users/sefayet/Desktop/Github/Machine_Learning-Deep_Learning-Courses-and-Paper-Publish/Thesis_Papers/Sarcasm_detection/04_outputs/tables
PREDICTIONS: /Users/sefayet/Desktop/Github/Machine_Learning-Deep_Learning-Courses-and-Paper-Publish/Thesis_Papers/Sarcasm_detection/04_outputs/predictions
REPORTS: /Users/sefayet/Desktop/Github/Machine_Learning-Deep_Learning-Courses-and-Paper-Publish/Thesis_Papers/Sarcasm_detection/04_outputs/reports
RUN_LOGS: /Users/sefayet/Desktop/Github/Machine_Learning-Deep_Learning-Courses-and-Paper-Publish/Thesis_Papers/Sarcasm_detec

In [5]:

# =========================
# OPTIONAL NORMALIZATION
# =========================

def normalize_text(text):
    # Keep baseline behavior unchanged unless USE_NORMALIZATION = True.
    # Add BanglaBERT-compatible normalization later if needed.
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    return text.strip()

def maybe_normalize_series(series):
    if USE_NORMALIZATION:
        return series.fillna("").astype(str).apply(normalize_text)
    return series.fillna("").astype(str)


In [6]:

# =========================
# TOKENIZER
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Loaded tokenizer for:", MODEL_NAME)


Loaded tokenizer for: csebuetnlp/banglabert


In [7]:

# =========================
# HELPERS
# =========================

def softmax_np(x):
    x = np.asarray(x)
    x = x - np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "precision_binary": float(p_bin),
        "recall_binary": float(r_bin),
        "f1_binary": float(f1_bin),
        "macro_f1": float(f1_macro),
    }

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def validate_input_df(df, dataset_name, split_name):
    required_cols = {TEXT_COL, LABEL_COL}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"{dataset_name} {split_name} is missing columns: {missing}")

def prepare_hf_datasets(train_df, val_df, test_df):
    for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        validate_input_df(df, "dataset", split_name)

    train_df = train_df[[TEXT_COL, LABEL_COL]].copy()
    val_df = val_df[[TEXT_COL, LABEL_COL]].copy()
    test_df = test_df[[TEXT_COL, LABEL_COL]].copy()

    train_df[TEXT_COL] = maybe_normalize_series(train_df[TEXT_COL])
    val_df[TEXT_COL] = maybe_normalize_series(val_df[TEXT_COL])
    test_df[TEXT_COL] = maybe_normalize_series(test_df[TEXT_COL])

    train_df = train_df.rename(columns={LABEL_COL: "label"})
    val_df = val_df.rename(columns={LABEL_COL: "label"})
    test_df = test_df.rename(columns={LABEL_COL: "label"})

    train_df["label"] = train_df["label"].astype(int)
    val_df["label"] = val_df["label"].astype(int)
    test_df["label"] = test_df["label"].astype(int)

    train_ds = Dataset.from_pandas(train_df, preserve_index=False)
    val_ds = Dataset.from_pandas(val_df, preserve_index=False)
    test_ds = Dataset.from_pandas(test_df, preserve_index=False)

    train_ds = train_ds.map(tokenize_batch, batched=True)
    val_ds = val_ds.map(tokenize_batch, batched=True)
    test_ds = test_ds.map(tokenize_batch, batched=True)

    train_ds = train_ds.remove_columns([TEXT_COL])
    val_ds = val_ds.remove_columns([TEXT_COL])
    test_ds = test_ds.remove_columns([TEXT_COL])

    train_ds.set_format("torch")
    val_ds.set_format("torch")
    test_ds.set_format("torch")

    return train_df, val_df, test_df, train_ds, val_ds, test_ds

def build_training_arguments(output_dir):
    common_kwargs = dict(
        output_dir=str(output_dir),
        save_strategy="epoch",
        logging_strategy="epoch",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
        seed=SEED,
        data_seed=SEED,
        fp16=USE_FP16,
        bf16=USE_BF16,
    )

    # compatibility across transformers versions
    try:
        return TrainingArguments(
            evaluation_strategy="epoch",
            **common_kwargs
        )
    except TypeError:
        return TrainingArguments(
            eval_strategy="epoch",
            **common_kwargs
        )

def dataset_label_stats(df, split_name):
    counts = df[LABEL_COL].value_counts().sort_index().to_dict()
    total = len(df)
    row = {"split": split_name, "n_rows": total}
    for k, v in counts.items():
        row[f"label_{k}_count"] = int(v)
        row[f"label_{k}_pct"] = round(100 * v / total, 2) if total else 0.0
    return row

def run_predictions(trainer, df_for_labels, ds, dataset_name, split_name):
    pred_output = trainer.predict(ds)
    logits = np.asarray(pred_output.predictions)
    probs = softmax_np(logits)
    preds = np.argmax(logits, axis=-1)
    labels = np.asarray(df_for_labels["label"])

    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    cm = confusion_matrix(labels, preds)
    report = classification_report(labels, preds, output_dict=True, zero_division=0)

    metrics = {
        "dataset": dataset_name,
        "split": split_name,
        "accuracy": float(acc),
        "precision_binary": float(p_bin),
        "recall_binary": float(r_bin),
        "f1_binary": float(f1_bin),
        "macro_f1": float(f1_macro),
        "n_examples": int(len(labels)),
    }

    pred_df = pd.DataFrame({
        "text": df_for_labels.get(TEXT_COL, pd.Series([""] * len(df_for_labels))).astype(str).values,
        "gold_label": labels,
        "pred_label": preds,
        "prob_label_0": probs[:, 0],
        "prob_label_1": probs[:, 1],
        "correct": (labels == preds).astype(int),
    })

    pred_df["dataset"] = dataset_name
    pred_df["split"] = split_name
    pred_df["model_name"] = MODEL_NAME
    pred_df["seed"] = SEED
    pred_df["max_length"] = MAX_LENGTH
    pred_df["epochs"] = EPOCHS
    pred_df["learning_rate"] = LR
    pred_df["use_normalization"] = USE_NORMALIZATION

    return metrics, cm, report, pred_df, pred_output

def flatten_classification_report(report_dict, dataset_name, split_name):
    rows = []
    for label_name, values in report_dict.items():
        if isinstance(values, dict):
            row = {"dataset": dataset_name, "split": split_name, "label_or_avg": label_name}
            row.update(values)
            rows.append(row)
    return pd.DataFrame(rows)

def safe_remove_dir(path_obj):
    if path_obj.exists() and path_obj.is_dir():
        shutil.rmtree(path_obj)


In [8]:

# =========================
# RUN ALL DATASETS
# =========================

all_summary_rows = []
all_confusions = {}
all_report_rows = []
all_data_stats = []
all_run_metadata = []

run_started_at = datetime.utcnow().isoformat()

for dataset_name in DATASET_NAMES:
    print("\n" + "=" * 100)
    print("RUNNING DATASET:", dataset_name)
    print("=" * 100)

    train_path = SPLITS / f"{dataset_name}_train.csv"
    val_path = SPLITS / f"{dataset_name}_val.csv"
    test_path = SPLITS / f"{dataset_name}_test.csv"

    if not train_path.exists():
        raise FileNotFoundError(f"Missing file: {train_path}")
    if not val_path.exists():
        raise FileNotFoundError(f"Missing file: {val_path}")
    if not test_path.exists():
        raise FileNotFoundError(f"Missing file: {test_path}")

    train_raw = pd.read_csv(train_path)
    val_raw = pd.read_csv(val_path)
    test_raw = pd.read_csv(test_path)

    print("Loaded shapes ->",
          "train:", train_raw.shape,
          "| val:", val_raw.shape,
          "| test:", test_raw.shape)

    all_data_stats.append({"dataset": dataset_name, **dataset_label_stats(train_raw, "train")})
    all_data_stats.append({"dataset": dataset_name, **dataset_label_stats(val_raw, "val")})
    all_data_stats.append({"dataset": dataset_name, **dataset_label_stats(test_raw, "test")})

    train_df, val_df, test_df, train_ds, val_ds, test_ds = prepare_hf_datasets(
        train_raw, val_raw, test_raw
    )

    checkpoint_dir = CHECKPOINTS / f"05_baseline_banglabert_{dataset_name}"
    if OVERWRITE_EXISTING_CHECKPOINTS and checkpoint_dir.exists():
        print("Removing old checkpoint dir:", checkpoint_dir)
        safe_remove_dir(checkpoint_dir)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    training_args = build_training_arguments(checkpoint_dir)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    print("Starting training...")
    train_output = trainer.train()
    print("Training completed for:", dataset_name)

    # Validation predictions
    val_metrics, val_cm, val_report, val_pred_df, val_pred_output = run_predictions(
        trainer, val_df, val_ds, dataset_name, "validation"
    )

    # Test predictions
    test_metrics, test_cm, test_report, test_pred_df, test_pred_output = run_predictions(
        trainer, test_df, test_ds, dataset_name, "test"
    )

    print("\nValidation metrics:", val_metrics)
    print("Validation confusion matrix:\n", val_cm)

    print("\nTest metrics:", test_metrics)
    print("Test confusion matrix:\n", test_cm)

    all_confusions[dataset_name] = {
        "validation": np.asarray(val_cm).tolist(),
        "test": np.asarray(test_cm).tolist(),
    }

    common_meta = {
        "model": "banglabert",
        "model_name": MODEL_NAME,
        "dataset": dataset_name,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "max_length": MAX_LENGTH,
        "seed": SEED,
        "use_normalization": USE_NORMALIZATION,
    }

    all_summary_rows.append({**common_meta, **val_metrics})
    all_summary_rows.append({**common_meta, **test_metrics})

    all_report_rows.append(flatten_classification_report(val_report, dataset_name, "validation"))
    all_report_rows.append(flatten_classification_report(test_report, dataset_name, "test"))

    # Save per-dataset metrics table
    per_dataset_summary_df = pd.DataFrame([
        row for row in all_summary_rows if row["dataset"] == dataset_name
    ]).sort_values(["dataset", "split"]).reset_index(drop=True)

    per_dataset_summary_path = TABLES / f"05_baseline_banglabert_{dataset_name}_summary.csv"
    per_dataset_summary_df.to_csv(per_dataset_summary_path, index=False)

    # Save per-dataset confusion matrices
    per_dataset_cm_path = TABLES / f"05_baseline_banglabert_{dataset_name}_confusion_matrices.json"
    with open(per_dataset_cm_path, "w", encoding="utf-8") as f:
        json.dump(all_confusions[dataset_name], f, ensure_ascii=False, indent=2)

    # Save reports
    per_dataset_report_df = pd.concat([
        flatten_classification_report(val_report, dataset_name, "validation"),
        flatten_classification_report(test_report, dataset_name, "test"),
    ], ignore_index=True)
    per_dataset_report_path = REPORTS / f"05_baseline_banglabert_{dataset_name}_classification_report.csv"
    per_dataset_report_df.to_csv(per_dataset_report_path, index=False)

    # Save predictions
    if SAVE_VAL_PREDICTIONS:
        val_pred_path = PREDICTIONS / f"05_baseline_banglabert_{dataset_name}_validation_predictions.csv"
        val_pred_df.to_csv(val_pred_path, index=False)

    if SAVE_TEST_PREDICTIONS:
        test_pred_path = PREDICTIONS / f"05_baseline_banglabert_{dataset_name}_test_predictions.csv"
        test_pred_df.to_csv(test_pred_path, index=False)

    # Save training log / metadata
    log_row = {
        "dataset": dataset_name,
        "checkpoint_dir": str(checkpoint_dir),
        "best_model_checkpoint": getattr(trainer.state, "best_model_checkpoint", None),
        "best_metric": getattr(trainer.state, "best_metric", None),
        "global_step": getattr(trainer.state, "global_step", None),
        "train_runtime": train_output.metrics.get("train_runtime"),
        "train_samples_per_second": train_output.metrics.get("train_samples_per_second"),
        "train_steps_per_second": train_output.metrics.get("train_steps_per_second"),
        "train_loss": train_output.metrics.get("train_loss"),
        "seed": SEED,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "max_length": MAX_LENGTH,
        "use_normalization": USE_NORMALIZATION,
        "run_started_at_utc": run_started_at,
        "run_finished_at_utc": datetime.utcnow().isoformat(),
    }
    all_run_metadata.append(log_row)

    print("\nSaved for", dataset_name)
    print("-", per_dataset_summary_path.name)
    print("-", per_dataset_cm_path.name)
    print("-", per_dataset_report_path.name)
    if SAVE_VAL_PREDICTIONS:
        print("-", val_pred_path.name)
    if SAVE_TEST_PREDICTIONS:
        print("-", test_pred_path.name)

print("\nAll datasets finished.")



RUNNING DATASET: ben_sarc_binary
Loaded shapes -> train: (20508, 4) | val: (2564, 4) | test: (2564, 4)


Map: 100%|██████████| 2564/2564 [00:00<00:00, 15361.82 examples/s]


Removing old checkpoint dir: ../03_models/checkpoints/05_baseline_banglabert_ben_sarc_binary


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 51980.24it/s]
ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	

Starting training...


...

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.54it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.42it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.81it/s]
There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 

Training completed for: banglasarc3_binary


/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



Validation metrics: {'dataset': 'banglasarc3_binary', 'split': 'validation', 'accuracy': 0.7668329177057357, 'precision_binary': 0.786096256684492, 'recall_binary': 0.7331670822942643, 'f1_binary': 0.7587096774193548, 'macro_f1': 0.7665683489629946, 'n_examples': 802}
Validation confusion matrix:
 [[321  80]
 [107 294]]

Test metrics: {'dataset': 'banglasarc3_binary', 'split': 'test', 'accuracy': 0.7369077306733167, 'precision_binary': 0.7461139896373057, 'recall_binary': 0.7182044887780549, 'f1_binary': 0.7318932655654383, 'macro_f1': 0.7368156658304548, 'n_examples': 802}
Test confusion matrix:
 [[303  98]
 [113 288]]

Saved for banglasarc3_binary
- 05_baseline_banglabert_banglasarc3_binary_summary.csv
- 05_baseline_banglabert_banglasarc3_binary_confusion_matrices.json
- 05_baseline_banglabert_banglasarc3_binary_classification_report.csv
- 05_baseline_banglabert_banglasarc3_binary_validation_predictions.csv
- 05_baseline_banglabert_banglasarc3_binary_test_predictions.csv

All datase

In [9]:

# =========================
# SAVE GLOBAL OUTPUTS
# =========================

summary_df = pd.DataFrame(all_summary_rows).sort_values(["dataset", "split"]).reset_index(drop=True)

reports_df = pd.concat(all_report_rows, ignore_index=True) if len(all_report_rows) > 0 else pd.DataFrame()

data_stats_df = pd.DataFrame(all_data_stats).reset_index(drop=True)
run_metadata_df = pd.DataFrame(all_run_metadata).reset_index(drop=True)

summary_path = TABLES / "05_baseline_banglabert_binary_summary_all_datasets.csv"
confusion_path = TABLES / "05_baseline_banglabert_binary_confusion_matrices_all_datasets.json"
reports_path = REPORTS / "05_baseline_banglabert_binary_classification_reports_all_datasets.csv"
data_stats_path = TABLES / "05_baseline_binary_dataset_label_stats.csv"
run_metadata_path = RUN_LOGS / "05_baseline_banglabert_run_metadata.csv"

summary_df.to_csv(summary_path, index=False)

with open(confusion_path, "w", encoding="utf-8") as f:
    json.dump(all_confusions, f, ensure_ascii=False, indent=2)

reports_df.to_csv(reports_path, index=False)
data_stats_df.to_csv(data_stats_path, index=False)
run_metadata_df.to_csv(run_metadata_path, index=False)

print("Saved global files:")
print("-", summary_path)
print("-", confusion_path)
print("-", reports_path)
print("-", data_stats_path)
print("-", run_metadata_path)


Saved global files:
- ../04_outputs/tables/05_baseline_banglabert_binary_summary_all_datasets.csv
- ../04_outputs/tables/05_baseline_banglabert_binary_confusion_matrices_all_datasets.json
- ../04_outputs/reports/05_baseline_banglabert_binary_classification_reports_all_datasets.csv
- ../04_outputs/tables/05_baseline_binary_dataset_label_stats.csv
- ../04_outputs/run_logs/05_baseline_banglabert_run_metadata.csv


In [10]:

# =========================
# DISPLAY RESULTS
# =========================

print("Summary metrics:")
display(summary_df)

print("\nDataset label stats:")
display(data_stats_df)

print("\nRun metadata:")
display(run_metadata_df)


Summary metrics:


,model,model_name,dataset,epochs,batch_size,learning_rate,weight_decay,warmup_ratio,max_length,seed,use_normalization,split,accuracy,precision_binary,recall_binary,f1_binary,macro_f1,n_examples
0,banglabert,csebuetnlp/banglabert,banglasarc3_binary,3,8,0.00002,0.01,0.1,128,42,False,test,0.736908,0.746114,0.718204,0.731893,0.736816,802
1,banglabert,csebuetnlp/banglabert,banglasarc3_binary,3,8,0.00002,0.01,0.1,128,42,False,validation,0.766833,0.786096,0.733167,0.758710,0.766568,802
2,banglabert,csebuetnlp/banglabert,banglasarc_binary,3,8,0.00002,0.01,0.1,128,42,False,test,0.986328,0.989637,0.974490,0.982005,0.985491,512
3,banglabert,csebuetnlp/banglabert,banglasarc_binary,3,8,0.00002,0.01,0.1,128,42,False,validation,0.984344,0.979487,0.979487,0.979487,0.983414,511
4,banglabert,csebuetnlp/banglabert,ben_sarc_binary,3,8,0.00002,0.01,0.1,128,42,False,test,0.799922,0.806375,0.789392,0.797793,0.799900,2564
5,banglabert,csebuetnlp/banglabert,ben_sarc_binary,3,8,0.00002,0.01,0.1,128,42,False,validation,0.801872,0.793627,0.815913,0.804615,0.801833,2564



Dataset label stats:


,dataset,split,n_rows,label_0_count,label_0_pct,label_1_count,label_1_pct
0,ben_sarc_binary,train,20508,10254,50.00,10254,50.00
1,ben_sarc_binary,val,2564,1282,50.00,1282,50.00
2,ben_sarc_binary,test,2564,1282,50.00,1282,50.00
3,banglasarc_binary,train,4089,2527,61.80,1562,38.20
4,banglasarc_binary,val,511,316,61.84,195,38.16
5,banglasarc_binary,test,512,316,61.72,196,38.28
6,banglasarc3_binary,train,6413,3207,50.01,3206,49.99
7,banglasarc3_binary,val,802,401,50.00,401,50.00
8,banglasarc3_binary,test,802,401,50.00,401,50.00



Run metadata:


,dataset,checkpoint_dir,best_model_checkpoint,best_metric,global_step,train_runtime,train_samples_per_second,train_steps_per_second,train_loss,seed,epochs,batch_size,learning_rate,max_length,use_normalization,run_started_at_utc,run_finished_at_utc
0,ben_sarc_binary,../03_models/checkpoints/05_baseline_banglaber...,../03_models/checkpoints/05_baseline_banglaber...,0.802221,7692,4775.9317,12.882,1.611,0.379660,42,3,8,0.00002,128,False,2026-03-26T11:40:02.995089,2026-03-26T13:02:14.314940
1,banglasarc_binary,../03_models/checkpoints/05_baseline_banglaber...,../03_models/checkpoints/05_baseline_banglaber...,0.983414,1536,1398.8980,8.769,1.098,0.083728,42,3,8,0.00002,128,False,2026-03-26T11:40:02.995089,2026-03-26T13:25:58.182082
2,banglasarc3_binary,../03_models/checkpoints/05_baseline_banglaber...,../03_models/checkpoints/05_baseline_banglaber...,0.766568,2406,2051.4899,9.378,1.173,0.459363,42,3,8,0.00002,128,False,2026-03-26T11:40:02.995089,2026-03-26T14:00:45.328231


In [11]:

# =========================
# QUICK ERROR SNAPSHOT
# =========================

example_error_tables = {}

for dataset_name in DATASET_NAMES:
    pred_path = PREDICTIONS / f"05_baseline_banglabert_{dataset_name}_test_predictions.csv"
    if pred_path.exists():
        pred_df = pd.read_csv(pred_path)
        err_df = pred_df[pred_df["correct"] == 0].copy()
        err_df["confidence"] = err_df[["prob_label_0", "prob_label_1"]].max(axis=1)
        err_df = err_df.sort_values("confidence", ascending=False).reset_index(drop=True)
        example_error_tables[dataset_name] = err_df.head(10)

for dataset_name, err_df in example_error_tables.items():
    print("\n" + "=" * 80)
    print("Top confident test errors:", dataset_name)
    print("=" * 80)
    display(err_df[["text", "gold_label", "pred_label", "prob_label_0", "prob_label_1", "confidence"]])



Top confident test errors: ben_sarc_binary


,text,gold_label,pred_label,prob_label_0,prob_label_1,confidence
0,চোটের কারণে দলের বাইরে চলে গেছেন এবি ডি ভিলিয়...,1,0,0.993937,0.006063,0.993937
1,অসম্ভব ভাল ছবি । এক কথায় অনবদ্য । বহুদিন পরে ...,1,0,0.993495,0.006505,0.993495
2,ঊনিশশো বাইশ এর পর ঊনিশশো তেইশ থেকে শুরু করে ঊন...,1,0,0.993382,0.006618,0.993382
3,ভালা লাগছে তুমি তাদের মনে রেখছো বলে,1,0,0.993260,0.006740,0.993260
4,আমি তো শিহরিত ।,1,0,0.993259,0.006741,0.993259
5,যে অসাধারন পারফরম্যান্স এইবার নিজের বাড়ি ফিরতে...,1,0,0.993219,0.006781,0.993219
6,রকেট গুলো আরো নিখুঁত ভাবে মারতে হবে ।,1,0,0.992971,0.007029,0.992971
7,ইয়ার্কির একটা সীমা থাকা উচিৎ,1,0,0.992942,0.007058,0.992942
8,পর্যাপ্ত আলো না থাকার কারণে পুরোপুরি বুঝতে পার...,1,0,0.992746,0.007254,0.992746
9,বাংলাদেশ জাতীয় দলের ভবিষ্যৎ কান্ডারি বলে মনে হয় ।,1,0,0.992684,0.007316,0.992684



Top confident test errors: banglasarc_binary


,text,gold_label,pred_label,prob_label_0,prob_label_1,confidence
0,অটোকারেক্ট এখনও মনে করে আমি দিনে 12 বার ‘হাঁস’...,1,0,0.999625,0.000375,0.999625
1,যাকে ইচ্ছা তাকে সেখানে আমি আপনি কাঁদলেও কি!,0,1,0.000609,0.999391,0.999391
2,"আপনারা সবাই একে অপরকেও জানেন, তাই না?",1,0,0.999381,0.000619,0.999381
3,তুমি না করলে কোনো কিছুই কাজে আসবে না.,1,0,0.999358,0.000642,0.999358
4,একটি বিপন্ন প্রজাতিতে শুভেচ্ছা,1,0,0.999354,0.000646,0.999354
5,অ্যালকোহল ‘প্রেরণ’ বোতামের আকার 89%বৃদ্ধি করে।,1,0,0.997298,0.002702,0.997298
6,"দুঃখিত, আমাদের হেল্পলাইন নাম্বার নেই।",0,1,0.012329,0.987671,0.987671



Top confident test errors: banglasarc3_binary


,text,gold_label,pred_label,prob_label_0,prob_label_1,confidence
0,ধর্ম গ্রন্থের রেফারেন্স টানার আগে ঘটনার প্রেক্...,1,0,0.961027,0.038973,0.961027
1,মনোযোগ দিয়ে প্রতিটা কথা শুনুন এবং সে অনুযায়ী...,1,0,0.958937,0.041063,0.958937
2,যে যেই যায়গার লোক তার টার্গেট তো সেখানেই থাকবে,1,0,0.953638,0.046362,0.953638
3,ইতিহাসের পাতায় লেখা থাকবে ৭১এর পরাজিত শক্তি মা...,1,0,0.951584,0.048416,0.951584
4,আপনার পোষ্ট গুলো অসাধারণ সবসময় সত্য কথা তুলে ধ...,1,0,0.951176,0.048824,0.951176
5,এইগুলারে আর এদের মা বাবাদের চিরিয়াখানায় রাখা উচিৎ,1,0,0.949316,0.050684,0.949316
6,কিছু বন্দী কে মুক্তি দেওয়া হোক,1,0,0.944189,0.055811,0.944189
7,হ্যাতের ভাষণ দুনিয়ার সবচেয়ে বিরক্তিকর জিনিস রি...,1,0,0.944030,0.055970,0.944030
8,চুপ কর সবাই পাপীর দল বাচ্চাডা ভালুপাশা প্রকাশ ...,1,0,0.942609,0.057391,0.942609
9,কমেন্ট বক্সে ১৬ বছরে ঘুমন্ত দেশপ্রেমিকদের ভীড় ...,1,0,0.940040,0.059960,0.940040
